# Download

In [1]:
# 在 Colab 单元格中运行
!pip install transformers sqlparse
!pip install accelerate xgrammar bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00


# import

In [1]:
import json
import time
from pathlib import Path
from typing import List

import torch
import sqlparse
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, LogitsProcessorList


# Settings

In [2]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
FILE_PATH_WIKI = "/content/drive/MyDrive/Colab Notebooks/NL2SQL/data/WikiSQL/wikisql_train_sample50.json"
OUT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/NL2SQL/results")
MAX_EXAMPLES = 50
MAX_NEW_TOKENS = 128
DEBUG_TOKENIZE = False  # print tokenization of column names for first few examples
DEVICE_PREFERENCE = "cuda"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Read Data

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import json
with open(FILE_PATH_WIKI, 'r') as f:
  data_wiki = json.load(f)

print("Total samples:", len(data_wiki))
print(data_wiki[0])

Total samples: 8421
{'instance_id': 'wikisql_dev_0', 'db': '1-10015132-11', 'question': 'What position does the player who played for butler cc (ks) play?', 'schema': {'table_names_original': ['table'], 'column_names_original': [[0, 'Player'], [0, 'No.'], [0, 'Nationality'], [0, 'Position'], [0, 'Years in Toronto'], [0, 'School/Club Team']], 'column_types': ['text', 'text', 'text', 'text', 'text', 'text']}, 'gold_sql_query': "SELECT Position FROM table WHERE School/Club Team = 'Butler CC (KS)'"}


# Utilities
## Build prompt

In [6]:
def build_schema_text(schema_obj: dict) -> str:
    # Convert WikiSQL schema block to a compact textual schema description.
    table_names = schema_obj.get("table_names_original", ["table"])
    col_entries = schema_obj.get("column_names_original", [])
    # group columns by table index
    table_cols = {}
    for entry in col_entries:
        if isinstance(entry, list) and len(entry) >= 2:
            tidx, cname = entry[0], entry[1]
        else:
            tidx, cname = 0, str(entry)
        table_cols.setdefault(tidx, []).append(cname)
    parts = []
    for tidx, cols in table_cols.items():
        tname = table_names[tidx] if tidx < len(table_names) else f"table{tidx}"
        cols_str = ", ".join(cols)
        parts.append(f"{tname}({cols_str})")
    return " ; ".join(parts)

def make_prompt(example: dict) -> str:
    schema_text = build_schema_text(example.get("schema", {}))
    question = example.get("question", "").strip()
    instance_id = example.get("instance_id", "")
    return (
        "Instruction: Translate the question to SQL.\n"
        f"Instance: {instance_id}\n"
        f"Schema: {schema_text}\n"
        f"Question: {question}\n"
        "Return only SQL:\n"
    )


### prompt making test

In [7]:
example_prompt = make_prompt(data_wiki[0])
print(example_prompt)

Instruction: Translate the question to SQL.
Instance: wikisql_dev_0
Schema: table(Player, No., Nationality, Position, Years in Toronto, School/Club Team)
Question: What position does the player who played for butler cc (ks) play?
Return only SQL:



## Save results

In [8]:
def save_results(file_name, results):
    out_path = OUT_DIR / file_name
    with open(str(out_path), "w", encoding="utf-8") as fout:
        json.dump(results, fout, ensure_ascii=False, indent=2)
    print(f"Saved {len(results)} records to {out_path}")

## Simple Evaluation

In [4]:
import json
import sqlparse

def normalize_sql(sql):
    if sql is None:
        return ""
    return " ".join(sql.strip().rstrip(";").lower().split())

def is_valid_sql(sql):
    try:
        return len(sqlparse.parse(sql)) > 0
    except:
        return False


def evaluate_auto(file_path):
    total = 0
    exact_match = 0
    empty_pred = 0
    valid_sql = 0
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
            if isinstance(data, dict):
                data = [data]
        except json.JSONDecodeError:
            f.seek(0)
            data = [json.loads(line) for line in f if line.strip()]

    for r in data:
        gold = r.get("gold_sql_query") or r.get("gold_sql") or ""
        pred = r.get("pred_sql") or r.get("predicted_sql") or ""

        total += 1

        if pred.strip() == "":
            empty_pred += 1

        if is_valid_sql(pred):
            valid_sql += 1

        if normalize_sql(gold) == normalize_sql(pred):
            exact_match += 1

    # -------- 统计 --------
    em_rate = exact_match / total if total > 0 else 0
    empty_rate = empty_pred / total if total > 0 else 0
    valid_rate = valid_sql / total if total > 0 else 0

    print("\n===== Evaluation Result =====")
    print(f"Total: {total}")
    print(f"Exact Match: {exact_match}/{total} = {em_rate:.4f}")
    print(f"Valid SQL: {valid_sql}/{total} = {valid_rate:.4f}")
    print(f"Empty Prediction: {empty_pred}/{total} = {empty_rate:.4f}")
    print("=============================\n")

    return {
        "total": total,
        "exact_match": exact_match,
        "exact_match_rate": em_rate,
        "valid_sql": valid_sql,
        "valid_sql_rate": valid_rate,
        "empty_pred": empty_pred,
        "empty_pred_rate": empty_rate,
    }

In [10]:
def print_tokenization(tokenizer, text: str, max_show: int = 200):
    ids = tokenizer.encode(text, add_special_tokens=False)
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"[tokenize] text: {text} -> {len(ids)} tokens")
    for i, t in enumerate(toks[:max_show]):
        print(f"  [{i:03d}] {t}")
    if len(ids) > max_show:
        print(f"  ... ({len(ids) - max_show} more tokens)")


## Load model & tokenizer

In [11]:
def load_model_and_tokenizer(model_name: str, device_preference: str = "cuda"):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    try:
        model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
        device = next(model.parameters()).device
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(model_name)
        if device_preference == "cuda" and torch.cuda.is_available():
            model = model.to("cuda")
            device = torch.device("cuda")
        else:
            model = model.to("cpu")
            device = torch.device("cpu")
    model.eval()
    return model, tokenizer, device

# Baseline

In [12]:
def generate_baseline(model, tokenizer, device, prompt: str, max_new_tokens: int):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    t0 = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    latency = time.time() - t0
    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return pred_sql, latency, int(gen_tokens.shape[0])

# XGrammar

In [13]:
import xgrammar as xgr
def escape_ebnf_literal(s: str) -> str:
    # 用 JSON 字符串形式生成 EBNF 引号内容，处理空格、标点、引号等
    return json.dumps(s)

def build_wikisql_ebnf(example: dict) -> str:
    schema = example["schema"]
    columns = [c[1] for c in schema["column_names_original"]]
    col_rules = " | ".join(escape_ebnf_literal(c) for c in columns)

    grammar = f"""
root ::= ws select_stmt ws

select_stmt ::= "SELECT" ws distinct_opt ws select_item ws "FROM" ws table_name ws where_opt ws order_opt ws limit_opt

distinct_opt ::= "" | "DISTINCT" ws

select_item ::= "*" | column_name | agg_func ws "(" ws "*" ws ")" | agg_func ws "(" ws column_name ws ")"
agg_func ::= "COUNT" | "MAX" | "MIN" | "SUM" | "AVG"

where_opt ::= "" | "WHERE" ws condition
condition ::= column_name ws op ws value

op ::= "=" | "!=" | "<" | ">" | "<=" | ">=" | "LIKE"

order_opt ::= "" | "ORDER" ws "BY" ws column_name ws order_dir
order_dir ::= "" | "ASC" | "DESC"

limit_opt ::= "" | "LIMIT" ws number

table_name ::= {escape_ebnf_literal("table")}
column_name ::= {col_rules}

value ::= quoted_string | number | bare_value
quoted_string ::= "'" quoted_body* "'"
quoted_body ::= [^'\\\\\\x00-\\x1F]
number ::= ["-"]? [0-9]+ ("." [0-9]+)?
bare_value ::= bare_char+
bare_char ::= [A-Za-z0-9_./()\\-]
ws ::= [ \\t\\n]*
"""
    return grammar.strip()


def build_wikisql_ebnf_soft(example: dict) -> str:
    schema = example["schema"]
    columns = [c[1] for c in schema["column_names_original"]]
    col_rules = " | ".join(escape_ebnf_literal(c) for c in columns)

    grammar =  """
root ::= ws "SELECT" ws anything ws "FROM" ws anything ws opt_where ws opt_order ws opt_limit ws

anything ::= [^;]+
opt_where ::= "" | "WHERE" ws anything
opt_order ::= "" | "ORDER" ws "BY" ws anything
opt_limit ::= "" | "LIMIT" ws number
number ::= [0-9]+
ws ::= [ \\t\\n]*
"""
    return grammar.strip()

In [15]:
def init_xgrammar(tokenizer, model_name):
  config = AutoConfig.from_pretrained(model_name)
  tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=config.vocab_size)
  grammar_compiler = xgr.GrammarCompiler(tokenizer_info, max_threads=8)
  return grammar_compiler

In [16]:
def generate_xgrammar(model, tokenizer, device, prompt: str, example: dict, grammar_compiler, max_new_tokens: int):
    ebnf = build_wikisql_ebnf(example)
    compiled_grammar = grammar_compiler.compile_grammar(ebnf)
    xgr_logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    t0 = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        logits_processor=[xgr_logits_processor],
    )
    latency = time.time() - t0
    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    pred_sql = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return pred_sql, latency, int(gen_tokens.shape[0])


# Run inference

In [17]:
def clean_sql(sql):
    sql = sql.replace("```sql", "").replace("```", "")
    return sql.strip()

def run_inference(
    mode: str,
    model_name: str,
    out_file_name: str,
    data,
    max_examples: int = 2,
    max_new_tokens: int = 128,
    debug_tokenize: bool = False,
):
    assert mode in ["baseline", "xgrammar"]

    count = min(len(data), max_examples)
    print(f"Loaded {len(data)} examples, running {count} examples in mode={mode}")

    model, tokenizer, device = load_model_and_tokenizer(model_name, DEVICE_PREFERENCE)

    grammar_compiler = None
    if mode == "xgrammar":
        grammar_compiler = init_xgrammar(tokenizer,model_name)

    results = []
    for i, ex in enumerate(data[:count]):
        prompt = make_prompt(ex)

        if debug_tokenize and i < 2:
            print(f"\n==== Example {i} instance_id={ex.get('instance_id')} ====")
            for ent in ex.get("schema", {}).get("column_names_original", []):
                cname = ent[1] if isinstance(ent, list) and len(ent) >= 2 else str(ent)
                print_tokenization(tokenizer, cname)
            print("==== end tokenization ====\n")

        try:
            if mode == "baseline":
                pred_sql, latency, n_tokens = generate_baseline(
                    model, tokenizer, device, prompt, max_new_tokens
                )
            else:
                pred_sql, latency, n_tokens = generate_xgrammar(
                    model, tokenizer, device, prompt, ex, grammar_compiler, max_new_tokens
                )
        except Exception as e:
            print(f"[error] example {i} failed: {e}")
            pred_sql, latency, n_tokens = "", 0.0, 0

        pred_sql=clean_sql(pred_sql)
        results.append({
            "instance_id": ex.get("instance_id"),
            "db": ex.get("db"),
            "question": ex.get("question"),
            "schema": ex.get("schema"),
            "gold_sql_query": ex.get("gold_sql_query"),
            "pred_sql": pred_sql,
        })

        if i % 10 == 0:
            print(f"[{i}] id={ex.get('instance_id')} pred_preview={pred_sql[:120]}")

    save_results(out_file_name,results)
    return results


## run baseline

In [28]:
file_name_baseline = "qwen_baseline_wikisql.jsonl"
baseline_results = run_inference(
    mode="baseline",
    model_name=MODEL_NAME,
    out_file_name=file_name_baseline,
    data=data_wiki,
    max_examples=50,
    max_new_tokens=MAX_NEW_TOKENS,
    debug_tokenize=DEBUG_TOKENIZE,
)

Loaded 8421 examples, running 50 examples in mode=baseline


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


==== Example 0 instance_id=wikisql_dev_0 ====
[tokenize] text: Player -> 1 tokens
  [000] Player
[tokenize] text: No. -> 2 tokens
  [000] No
  [001] .
[tokenize] text: Nationality -> 2 tokens
  [000] National
  [001] ity
[tokenize] text: Position -> 1 tokens
  [000] Position
[tokenize] text: Years in Toronto -> 3 tokens
  [000] Years
  [001] Ġin
  [002] ĠToronto
[tokenize] text: School/Club Team -> 4 tokens
  [000] School
  [001] /
  [002] Club
  [003] ĠTeam
==== end tokenization ====

[0] id=wikisql_dev_0 pred_preview=```sql
SELECT Position FROM table WHERE School/Club Team = 'Butler CC (KS)' AND Nationality = 'KS';
```

==== Example 1 instance_id=wikisql_dev_1 ====
[tokenize] text: Player -> 1 tokens
  [000] Player
[tokenize] text: No. -> 2 tokens
  [000] No
  [001] .
[tokenize] text: Nationality -> 2 tokens
  [000] National
  [001] ity
[tokenize] text: Position -> 1 tokens
  [000] Position
[tokenize] text: Years in Toronto -> 3 tokens
  [000] Years
  [001] Ġin
  [002] ĠToronto
[tok

In [29]:
baseline_result_path = OUT_DIR / file_name_baseline
evaluate_auto(baseline_result_path)


===== Evaluation Result =====
Total: 50
Exact Match: 0/50 = 0.0000
Valid SQL: 50/50 = 1.0000
Empty Prediction: 0/50 = 0.0000



{'total': 50,
 'exact_match': 0,
 'exact_match_rate': 0.0,
 'valid_sql': 50,
 'valid_sql_rate': 1.0,
 'empty_pred': 0,
 'empty_pred_rate': 0.0}

## run constrained decoding with XGrammar

In [30]:
file_name_xgrammar = "qwen_xgrammar_wikisql.jsonl"
xgrammar_results = run_inference(
    mode="xgrammar",
    model_name=MODEL_NAME,
    out_file_name=file_name_xgrammar,
    data=data_wiki,
    max_examples=50,
    max_new_tokens=MAX_NEW_TOKENS,
    debug_tokenize=DEBUG_TOKENIZE,
)

Loaded 8421 examples, running 50 examples in mode=xgrammar


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


==== Example 0 instance_id=wikisql_dev_0 ====
[tokenize] text: Player -> 1 tokens
  [000] Player
[tokenize] text: No. -> 2 tokens
  [000] No
  [001] .
[tokenize] text: Nationality -> 2 tokens
  [000] National
  [001] ity
[tokenize] text: Position -> 1 tokens
  [000] Position
[tokenize] text: Years in Toronto -> 3 tokens
  [000] Years
  [001] Ġin
  [002] ĠToronto
[tokenize] text: School/Club Team -> 4 tokens
  [000] School
  [001] /
  [002] Club
  [003] ĠTeam
==== end tokenization ====

[0] id=wikisql_dev_0 pred_preview=SELECT Position FROM table WHERE School/Club Team = 'Butler CC' ORDER BY Years in Toronto DESC LIMIT 1

==== Example 1 instance_id=wikisql_dev_1 ====
[tokenize] text: Player -> 1 tokens
  [000] Player
[tokenize] text: No. -> 2 tokens
  [000] No
  [001] .
[tokenize] text: Nationality -> 2 tokens
  [000] National
  [001] ity
[tokenize] text: Position -> 1 tokens
  [000] Position
[tokenize] text: Years in Toronto -> 3 tokens
  [000] Years
  [001] Ġin
  [002] ĠToronto
[toke

In [31]:
xgrammar_result_path = OUT_DIR / file_name_xgrammar
evaluate_auto(xgrammar_result_path)


===== Evaluation Result =====
Total: 50
Exact Match: 0/50 = 0.0000
Valid SQL: 50/50 = 1.0000
Empty Prediction: 0/50 = 0.0000



{'total': 50,
 'exact_match': 0,
 'exact_match_rate': 0.0,
 'valid_sql': 50,
 'valid_sql_rate': 1.0,
 'empty_pred': 0,
 'empty_pred_rate': 0.0}

In [5]:
result_path = "results/result_wikisql_non-thinking_zero_xgrammar.json"
evaluate_auto(result_path)


===== Evaluation Result =====
Total: 50
Exact Match: 9/50 = 0.1800
Valid SQL: 50/50 = 1.0000
Empty Prediction: 0/50 = 0.0000



{'total': 50,
 'exact_match': 9,
 'exact_match_rate': 0.18,
 'valid_sql': 50,
 'valid_sql_rate': 1.0,
 'empty_pred': 0,
 'empty_pred_rate': 0.0}

In [13]:
result_path = "results/result_wikisql_non-thinking_zero_xgrammar_new.json"
evaluate_auto(result_path)


===== Evaluation Result =====
Total: 50
Exact Match: 12/50 = 0.2400
Valid SQL: 49/50 = 0.9800
Empty Prediction: 1/50 = 0.0200



{'total': 50,
 'exact_match': 12,
 'exact_match_rate': 0.24,
 'valid_sql': 49,
 'valid_sql_rate': 0.98,
 'empty_pred': 1,
 'empty_pred_rate': 0.02}

In [6]:
result_path = "results/result_wikisql_non-thinking_zero_none.json"
evaluate_auto(result_path)


===== Evaluation Result =====
Total: 50
Exact Match: 10/50 = 0.2000
Valid SQL: 50/50 = 1.0000
Empty Prediction: 0/50 = 0.0000



{'total': 50,
 'exact_match': 10,
 'exact_match_rate': 0.2,
 'valid_sql': 50,
 'valid_sql_rate': 1.0,
 'empty_pred': 0,
 'empty_pred_rate': 0.0}

In [7]:
result_path = "results/result_wikisql_non-thinking_zero_outlines.json"
evaluate_auto(result_path)


===== Evaluation Result =====
Total: 50
Exact Match: 10/50 = 0.2000
Valid SQL: 50/50 = 1.0000
Empty Prediction: 0/50 = 0.0000



{'total': 50,
 'exact_match': 10,
 'exact_match_rate': 0.2,
 'valid_sql': 50,
 'valid_sql_rate': 1.0,
 'empty_pred': 0,
 'empty_pred_rate': 0.0}

In [10]:
result_path = "results/result_wikisql_thinking_zero_xgrammar.json"
evaluate_auto(result_path)


===== Evaluation Result =====
Total: 50
Exact Match: 4/50 = 0.0800
Valid SQL: 36/50 = 0.7200
Empty Prediction: 14/50 = 0.2800



{'total': 50,
 'exact_match': 4,
 'exact_match_rate': 0.08,
 'valid_sql': 36,
 'valid_sql_rate': 0.72,
 'empty_pred': 14,
 'empty_pred_rate': 0.28}

In [14]:
result_path = "results/result_wikisql_thinking_zero_xgrammar (1).json"
evaluate_auto(result_path)


===== Evaluation Result =====
Total: 50
Exact Match: 0/50 = 0.0000
Valid SQL: 40/50 = 0.8000
Empty Prediction: 10/50 = 0.2000



{'total': 50,
 'exact_match': 0,
 'exact_match_rate': 0.0,
 'valid_sql': 40,
 'valid_sql_rate': 0.8,
 'empty_pred': 10,
 'empty_pred_rate': 0.2}

In [8]:
result_path = "results/result_wikisql_thinking_zero_none.json"
evaluate_auto(result_path)


===== Evaluation Result =====
Total: 50
Exact Match: 16/50 = 0.3200
Valid SQL: 48/50 = 0.9600
Empty Prediction: 2/50 = 0.0400



{'total': 50,
 'exact_match': 16,
 'exact_match_rate': 0.32,
 'valid_sql': 48,
 'valid_sql_rate': 0.96,
 'empty_pred': 2,
 'empty_pred_rate': 0.04}

In [9]:
result_path = "results/result_wikisql_thinking_zero_outlines.json"
evaluate_auto(result_path)


===== Evaluation Result =====
Total: 50
Exact Match: 4/50 = 0.0800
Valid SQL: 50/50 = 1.0000
Empty Prediction: 0/50 = 0.0000



{'total': 50,
 'exact_match': 4,
 'exact_match_rate': 0.08,
 'valid_sql': 50,
 'valid_sql_rate': 1.0,
 'empty_pred': 0,
 'empty_pred_rate': 0.0}

In [ ]:
def generate_plain(
    model,
    tokenizer,
    prompt: str,
    generation_kwargs: Dict[str, Any],
):
    """
    Plain Hugging Face generation without constrained decoding.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    start_time = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_kwargs,
        )

    latency_ms = (time.perf_counter() - start_time) * 1000.0

    generated_ids = outputs[0][input_len:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=False)
    num_tokens_generated = int(generated_ids.shape[0])

    return text, num_tokens_generated, latency_ms